In [ ]:
# Auto-Fix Working Directory 
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory set to:", os.getcwd())


# Nonlinear Regression — Part 1
### Enzyme Kinetics Data Processing — Alkaline Phosphatase Assay

**Author:** Omar Hamouda

**Environment:** Python 3.x | Jupyter Notebook | Anaconda

---
## 📌 Overview
This notebook processes **raw enzyme kinetics data** from an alkaline
phosphatase assay. We will:
1. Import raw absorbance data (ΔA₄₀₅) into a pandas DataFrame.
2. Set time as the index.
3. Calculate initial rates (slopes) at each substrate concentration.
4. Convert slopes to initial velocities using the extinction coefficient.
5. Export the processed data to a CSV file for Part 2.

## 📚 Table of Contents
1. Importing Libraries (os, pandas, scipy.stats)
2. Loading & Inspecting the Data
3. Setting Time as the Index
4. Inspecting Data with `df.plot()`
5. Creating a Second DataFrame (MM_df)
6. Calculating Slopes with `linregress`
7. Calculating Initial Velocities
8. Exporting to CSV
9. Exercise: Chymotrypsin Kinetics

In [ ]:
# Import the libraries we need
import os                          # For file paths
import pandas as pd                # For DataFrames
from scipy import stats            # For linear regression
import matplotlib.pyplot as plt    # For data inspection plots

In [ ]:
# Build the path to the data file
datafile = os.path.join('data', 'AP_kin.csv')
print(datafile)

# Verify it exists
if os.path.exists(datafile):
    print("File found ✅")
else:
    print("File not found ❌")

In [ ]:
# Load the raw kinetics data into a DataFrame
AP_kin_df = pd.read_csv(datafile)

# Check column names FIRST (lesson from KeyError!)
print("Columns:", AP_kin_df.columns.tolist())

# Display the first 5 rows
AP_kin_df.head()

In [ ]:
# Set 'Time (min)' as the index (modify in place)
AP_kin_df.set_index('Time (min)', inplace=True)

# Verify the change
AP_kin_df.head()

In [ ]:
# Check the datatype of the index
AP_kin_df.index.dtype

In [ ]:
# Quick visual inspection with pandas' built-in plot
AP_kin_df.plot(marker='o', figsize=(10, 6))
plt.xlabel('Time (min)')
plt.ylabel('$A_{405}$')
plt.show()

In [ ]:
# Create a new DataFrame for Michaelis-Menten analysis
MM_df = pd.DataFrame()

# Add substrate concentrations (the column names from AP_kin_df)
MM_df['pNPP (mM)'] = AP_kin_df.columns

# Display
MM_df

In [ ]:
# Create an empty list to hold the slopes
slopes = []

# Loop through each column (each substrate concentration)
for column in AP_kin_df.columns:
    # Linear regression: time (index) vs absorbance (column)
    slope, _, _, _, _ = stats.linregress(AP_kin_df.index, AP_kin_df[column])
    slopes.append(slope)

# Display the slopes
print(slopes)

In [ ]:
# Add slopes to the MM_df DataFrame
MM_df['slopes'] = slopes

# Display
MM_df

In [ ]:
# Calculate initial velocities = slope / extinction_coefficient
MM_df['initial velocities'] = MM_df['slopes'] / 0.015

# Display
MM_df

In [ ]:
# Build the output path
outputfile = os.path.join('data', 'MM_data.csv')

# Save to CSV
MM_df.to_csv(outputfile)

print("File saved ✅")

In [ ]:
# Verify the file was created
with open(outputfile, 'r') as f:
    content = f.read()

print(content)

---
## 🧪 Exercise: Chymotrypsin Kinetics

### 🎯 Goal
Process raw kinetics data from a **chymotrypsin assay** using the
same workflow as the alkaline phosphatase example.

### 📌 Key Differences
- **Data source:** Converted from Excel (manual cleanup required).
- **Extinction coefficient:** **0.01832 µM⁻¹cm⁻¹** (vs 0.015 for ALP).
- **Substrate:** p-Nitrophenyl Acetate (pNPA) — different from pNPP.

### 🛠️ Workflow
1. Load the cleaned CSV data.
2. Set time as the index.
3. Calculate slopes at each substrate concentration.
4. Convert slopes to initial velocities.
5. Export to `chymo_MM_data.csv`.

In [ ]:
# Build the path to the chymotrypsin data
chymo_file = os.path.join('data', 'chymotrypsin_kinetics.csv')
print(chymo_file)

# Verify it exists (best practice)
if os.path.exists(chymo_file):
    print("File found ✅")
else:
    print("File not found ❌")

### Loading the Chymotrypsin Data

We load the cleaned CSV into a DataFrame using the same pattern
from Lesson 05. **Always check `df.columns`** before referencing
any column name — this is the lesson we learned from the KeyError.

In [ ]:
# Load the chymotrypsin kinetics data
chymo_rates_df = pd.read_csv(chymo_file)

# Check column names FIRST
print("Columns:", chymo_rates_df.columns.tolist())

# Display the first 5 rows
chymo_rates_df.head()

### Setting Time as the Index

We set `'Time (sec)'` as the index — this mirrors the alkaline
phosphatase workflow. The index becomes our X values for the
linear regression.

We also check the **datatype** of the index to make sure it's numeric.

In [ ]:
# Set 'Time (sec)' as the index
chymo_rates_df.set_index('Time (sec)', inplace=True)

# Check the datatype of the index
print("Index dtype:", chymo_rates_df.index.dtype)

# Display the first 5 rows
chymo_rates_df.head()

### Visual Inspection

Before any calculations, we **look at the data** using pandas'
built-in `.plot()`. This quick visual check helps us:
- Confirm the data makes sense.
- Detect any anomalies (outliers, wrong scaling).
- Verify that all curves start near the origin.

In [ ]:
# Quick visual inspection with pandas' built-in plot
chymo_rates_df.plot(marker='o', figsize=(10, 6))
plt.xlabel('Time (sec)')
plt.ylabel('$A_{405}$')
plt.show()

### Creating the Michaelis-Menten DataFrame

We create a new, clean DataFrame (`chymo_MM_df`) to hold:
- Substrate concentrations (`pNPA (mM)`).
- Slopes (computed next).
- Initial velocities (computed after).

This **separates raw data from processed results** — a clean
data-analysis practice.

In [ ]:
# Create a new DataFrame for Michaelis-Menten analysis
chymo_MM_df = pd.DataFrame()

# Add substrate concentrations (the column names from chymo_rates_df)
chymo_MM_df['pNPA (mM)'] = chymo_rates_df.columns

# Display
chymo_MM_df

### Calculating Slopes

For each substrate concentration (each column), we perform a
**linear regression** of absorbance vs time. The **slope** is
proportional to the initial reaction rate.

We use `linregress()` from `scipy.stats` and ignore all outputs
except the slope (using `_` placeholders).

In [ ]:
# Create an empty list to hold the slopes
slopes = []

# Loop through each substrate concentration column
for column in chymo_rates_df.columns:
    slope, _, _, _, _ = stats.linregress(chymo_rates_df.index, chymo_rates_df[column])
    slopes.append(slope)

# Display the list of slopes
print(slopes)

### Adding Slopes to the DataFrame

We append the slopes to `chymo_MM_df` as a new column — keeping
the data organized alongside the substrate concentrations.

In [ ]:
# Add slopes to the DataFrame
chymo_MM_df['slopes'] = slopes

# Display
chymo_MM_df

### Converting Slopes to Initial Velocities

Using the **Beer-Lambert law**: velocity = slope / ε


For chymotrypsin under these conditions:
- **ε = 18,320 M⁻¹cm⁻¹ = 0.01832 µM⁻¹cm⁻¹**

This is a **larger** extinction coefficient than alkaline phosphatase,
which is why the velocities will be numerically smaller for the same slopes.

In [ ]:
# Calculate initial velocities = slope / extinction_coefficient
chymo_MM_df['initial velocities'] = chymo_MM_df['slopes'] / 0.01832

# Display
chymo_MM_df

### Setting the Index and Exporting

Two final steps:
1. **Set `pNPA (mM)` as the index** — cleaner DataFrame.
2. **Export to CSV** — the file `chymo_MM_data.csv` will be used
   in the next lesson (Nonlinear Regression Part 2).

In [ ]:
# Set 'pNPA (mM)' as the index
chymo_MM_df.set_index('pNPA (mM)', inplace=True)

# Display
chymo_MM_df

In [ ]:
# Build the output path
outputfile_chymo = os.path.join('data', 'chymo_MM_data.csv')

# Save to CSV
chymo_MM_df.to_csv(outputfile_chymo)

print("File saved ✅")

In [ ]:
# Verify the file was created correctly
with open(outputfile_chymo, 'r') as f:
    content = f.read()

print(content)

---
## 📝 Summary & Key Takeaways

In this notebook, we processed **raw enzyme kinetics data** from
**two different assays** — alkaline phosphatase (ALP) and
chymotrypsin — transforming time-series absorbance measurements
into **initial velocities** ready for nonlinear regression.

### 🧪 Assays Processed

| Assay | Enzyme | Substrate | Product | ε (µM⁻¹cm⁻¹) | Time Unit |
|-------|--------|-----------|---------|---------------|-----------|
| **ALP** | Alkaline Phosphatase | pNPP | p-Nitrophenol | 0.015 | min |
| **Chymotrypsin** | Chymotrypsin | pNPA | p-Nitrophenol | 0.01832 | sec |

### 🛠️ What We Covered

- **Imports** — `os`, `pandas`, `scipy.stats`, `matplotlib.pyplot`.
- **Loading Data** — `pd.read_csv()` with `os.path.join()`.
- **Setting the Index** — `set_index('Time', inplace=True)`.
- **Checking Datatype** — `.index.dtype` must be `float64`.
- **Visual Inspection** — `df.plot(marker='o')` for a quick check.
- **Creating New DataFrames** — `MM_df` and `chymo_MM_df`.
- **Calculating Slopes** — `stats.linregress()` in a `for` loop.
- **Converting to Velocities** — `slope / extinction_coefficient`.
- **Exporting Data** — `df.to_csv()` for use in Part 2.

### 📊 Results Produced

| File | Content |
|------|---------|
| **`data/MM_data.csv`** | ALP kinetics — 11 substrate concentrations |
| **`data/chymo_MM_data.csv`** | Chymotrypsin kinetics — 9 substrate concentrations |

### 🧮 Key Formulas
Slope = ΔA₄₀₅ / Δt (from linregress)
Velocity = Slope / ε (Beer-Lambert law)



---

## ⚠️ Important Notes

### On Enzyme Kinetics
- **Initial velocity** = the slope of the **linear portion** of the
  progress curve (first ~10% of substrate consumption).
- The **extinction coefficient** depends on **pH, buffer, temperature,
  and wavelength**. Always use the correct value.
- **Units matter:** a velocity in µM/min ≠ µM/sec. Be consistent.

### On Data Processing
- **Always check `df.columns`** after `read_csv()` — avoids `KeyError`.
- **Verify the datatype** of the index before any calculation.
- Use **`_` placeholders** to ignore unused return values from
  `linregress` — cleaner code.
- **`inplace=True`** modifies the DataFrame permanently.

### On File Output
- `df.to_csv()` includes the index by default.
- Use `index=False` to omit it if not needed.
- **Verify the file** after writing with a quick `open()` + `read()`.

### On Writing Robust Code
- **Don't hard-code values** — extract them from data (`.min()`, `.max()`).
- **Separate raw data from processed results** (use different DataFrames).
- **Document each step** with Markdown — your future self will thank you.

---

## 🔜 Next Steps

### Immediate (Lesson 09 — Nonlinear Regression Part 2)
- Load `MM_data.csv` and `chymo_MM_data.csv`.
- Use `scipy.optimize.curve_fit()` to fit the **Michaelis-Menten equation**.
- Estimate **Vmax** and **Km** for both enzymes.
- Plot the fitted curve with experimental data.

### Extended
- **Compare kinetics** — how does chymotrypsin's Km compare to ALP's?
- **Study inhibitors** — repeat with an inhibitor present.
- **Vary conditions** — pH, temperature, ionic strength.
- **Automate** — write a function to process any kinetics CSV.

---

## 🧬 Biological Context

### Alkaline Phosphatase (ALP)
An enzyme that **removes phosphate groups** (dephosphorylation) at
alkaline pH. Widely used as a **reporter enzyme** in molecular biology
(for example, in ELISA and Western blotting).

### Chymotrypsin
A **serine protease** that cleaves peptide bonds, primarily after
aromatic residues (Phe, Trp, Tyr). Used as a model system for
studying **proteolytic mechanisms** and **enzyme inhibition**.

### Why Enzyme Kinetics Matters
Understanding kinetics lets you:
- **Determine Km and Vmax** — quantifying enzyme efficiency.
- **Study inhibitors** — for drug discovery.
- **Compare variants** — wild-type vs. mutant enzymes.
- **Optimize assays** — choosing the right conditions.
- **Model metabolism** — building predictive models.

### Connection to Real Research
This workflow — raw data → initial rates → kinetic parameters — is
**exactly what researchers do daily** in biochemistry labs around
the world. The skills here are directly transferable to:
- Pharmaceutical R&D (drug screening).
- Biotechnology (enzyme engineering).
- Clinical diagnostics (measuring enzyme levels).
- Basic research (understanding catalytic mechanisms).

In [ ]:
# Final verification: read back the saved file
outputfile = os.path.join('data', 'MM_data.csv')

with open(outputfile, 'r') as f:
    content = f.read()

print(content)

In [ ]:
# Re-save without the index column (cleaner)
outputfile = os.path.join('data', 'MM_data.csv')
MM_df.to_csv(outputfile, index=False)

print("Re-saved ✅")